In [ ]:
folder = "results"
seed = 42
shot_wise = False

In [ ]:
import os
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
import random

random.seed(seed)

data = []
for file in os.listdir(folder):
    with open(os.path.join(folder, file), "r") as f:
        if file.endswith(".json"):
            data.append(json.load(f))
            f.close()

In [ ]:
def normalise(distribution):
    total = sum(distribution.values())
    normalised = {}
    for key, value in distribution.items():
        normalised[key] = value / total
        
    assert math.isclose(sum(normalised.values()), 1.0, rel_tol=1e-9), "Distribution does not sum to 1"
    return normalised

In [ ]:
def hellinger_distance(dict1, dict2):
    """
    Computes the Hellinger distance between two discrete probability distributions.
    
    The Hellinger distance is defined as:
        H(P, Q) = (1/√2) * sqrt( sum_x (sqrt(P(x)) - sqrt(Q(x)))^2 )
        
    Parameters:
        dict1 (dict): A dictionary with bistrings (as strings) as keys and probabilities as values.
        dict2 (dict): A dictionary with bistrings (as strings) as keys and probabilities as values.
        
    Returns:
        float: The Hellinger distance between the two distributions.
    """
    # Get the union of keys from both dictionaries
    keys = set(dict1.keys()).union(dict2.keys())
    
    # Compute the sum of squared differences between the square roots of the probabilities
    sum_sq_diff = 0.0
    for key in keys:
        p = dict1.get(key, 0)
        q = dict2.get(key, 0)
        sum_sq_diff += (math.sqrt(p) - math.sqrt(q)) ** 2
    
    # Normalize by 1/√2 and return the Hellinger distance
    return math.sqrt(sum_sq_diff) / math.sqrt(2)


def stddev(bitstring_probs: dict) -> float:
    """
    Computes the standard deviation of a probability distribution where the outcomes are given
    as bitstrings (interpreted as binary numbers) and their corresponding probabilities.
    
    Args:
        bitstring_probs (dict): A dictionary where keys are bitstrings (e.g., "101") and
                                values are the probabilities associated with each outcome.
                                
    Returns:
        float: The standard deviation of the distribution.
    """
    # Compute the mean (expected value)
    mean = sum(int(bitstring, 2) * prob for bitstring, prob in bitstring_probs.items())
    
    # Compute the variance
    variance = sum(((int(bitstring, 2) - mean) ** 2) * prob for bitstring, prob in bitstring_probs.items())
    
    # Standard deviation is the square root of the variance
    return math.sqrt(variance)

def total_variation_distance(dict1, dict2):
    """
    Computes the total variation distance between two discrete probability distributions.
    
    The total variation distance is defined as:
        TV(P, Q) = 0.5 * sum_x |P(x) - Q(x)|
        
    Parameters:
        dict1 (dict): A dictionary with bistrings (as strings) as keys and probabilities as values.
        dict2 (dict): A dictionary with bistrings (as strings) as keys and probabilities as values.
        
    Returns:
        float: The total variation distance between the two distributions.
    """
    # Get the union of keys from both dictionaries
    keys = set(dict1.keys()).union(dict2.keys())
    
    # Compute the sum of absolute differences
    sum_abs_diff = 0.0
    for key in keys:
        p = dict1.get(key, 0)
        q = dict2.get(key, 0)
        sum_abs_diff += abs(p - q)
    
    # Normalize by 0.5 and return the total variation distance
    return sum_abs_diff / 2

def entropy(bitstring_probs: dict) -> float:
    """
    Computes the Shannon entropy of a probability distribution where the outcomes are given
    as bitstrings (interpreted as binary numbers) and their corresponding probabilities.
    
    Args:
        bitstring_probs (dict): A dictionary where keys are bitstrings (e.g., "101") and
                                values are the probabilities associated with each outcome.
                                
    Returns:
        float: The Shannon entropy of the distribution.
    """
    # Compute the entropy using the formula: H(X) = -sum(p(x) * log2(p(x)))
    return -sum(prob * math.log2(prob) for prob in bitstring_probs.values() if prob > 0)

In [ ]:
distances = {
    "hellinger": hellinger_distance,
    "TVD": total_variation_distance,
}
measures = {
    "stddev": stddev,
    "entropy": entropy,
}

In [ ]:
ds = []

In [ ]:
# for exp in data:
    
#     for provider, bs_ls in exp["metadata"]["backends"].items():
#         __bs = []
#         for bs in bs_ls:
#             if bs not in __bs:
#                 __bs.append(bs)
    
#     d = {
#         "circuit": exp["metadata"]["circuit"],
#         "size": exp["metadata"]["circuit_size"],
#         "backends": "/".join(__bs),
#         "backends_len": len(__bs),
#     }

#     gt = normalise(exp["metadata"]["ground_truth"])
    
#     last_cumulated_distribution = None
    
#     _data = exp["data"].copy()
#     for i, s in enumerate(_data): 
#         _d = d.copy()
#         _d["it"] = i

#         agg_dist = s
        
#         if last_cumulated_distribution is not None:
#             for k in last_cumulated_distribution:
#                 agg_dist[k] = agg_dist.get(k, 0) + last_cumulated_distribution[k]
                
#         _d["shots"] = sum(agg_dist.values())

#         probs = normalise(agg_dist)
        
#         _d["error"] = hellinger_distance(probs, gt)
#         _d["hellinger"] = hellinger_distance(probs, gt)
#         _d["stddev"] = stddev(probs)
        
#         if last_cumulated_distribution is None:
#             _d["delta-hellinger"] = None
#             _d["delta-stddev"] = None
#         else:
#             last_probs = normalise(last_cumulated_distribution)
            
#             _d["delta-hellinger"] = hellinger_distance(last_probs, probs)
#             _d["delta-stddev"] = stddev(last_probs) - stddev(probs)
        
#         ds.append(_d)
        
#         last_cumulated_distribution = agg_dist.copy()

In [ ]:
for exp in data:
    if exp["metadata"]["seed"] == 24:
        continue
    d = {
        "circuit": exp["metadata"]["circuit"],
        "size": exp["metadata"]["circuit_size"],
        "seed": exp["metadata"]["seed"],
    }
    
    gt = normalise(exp["metadata"]["ground_truth"])
    
    last_cumulated_distribution = {}
    agg_dist = {}
    
    _data = exp["individual"].copy()
    for i, s in enumerate(_data): 
        _d = d.copy()
        _d["it"] = i
        
        for provider, bs_ls in s.items():
            if provider not in agg_dist:
                agg_dist[provider] = {}
            for b, r in bs_ls.items():
                __d = _d.copy()
                __d["backends"] = b
                __d["backends_len"] = 1
                if b not in agg_dist[provider]:
                    agg_dist[provider][b] = {}
                    
                _dist = r[0]['result']
                for k, v in _dist.items():
                    agg_dist[provider][b][k] = agg_dist[provider][b].get(k, 0) + v

                __d["shots"] = sum((agg_dist[provider][b]).values())
                
                probs = normalise(agg_dist[provider][b])
                
                __d["error"] = hellinger_distance(probs, gt)
                
                
                for d_name, d_fun in distances.items():
                    __d[d_name] = d_fun(probs, gt)
                for m_name, m_fun in measures.items():
                    __d[m_name] = m_fun(probs)
                
                if provider not in last_cumulated_distribution or b not in last_cumulated_distribution[provider]:
                    for d_name in distances.keys():
                        __d["delta-"+d_name] = None
                    for m_name in measures.keys():
                        __d["delta-"+m_name] = None
                else:
                    last_probs = normalise(last_cumulated_distribution[provider][b])
                    
                    for d_name, d_fun in distances.items():
                        __d["delta-"+d_name] = d_fun(last_probs, probs)
                    for m_name, m_fun in measures.items():
                        __d["delta-"+m_name] = m_fun(last_probs) - m_fun(probs)
                    
                ds.append(__d)

                last_cumulated_distribution = deepcopy(agg_dist)
       

In [ ]:
import random
from collections import Counter

def resample_freq_dict(freq_dict, n_samples):
    """
    Resample a frequency dictionary of bitstrings.

    Parameters:
        freq_dict (dict): A dictionary where keys are bitstrings (e.g., '0101') 
                          and values are their frequencies (non-negative integers).
        n_samples (int): The number of samples to draw, must be <= sum(freq_dict.values()).

    Returns:
        dict: A new dictionary with bitstrings and their frequency counts 
              obtained from sampling without replacement.
    """
    # Expand the dictionary into a list with each bitstring repeated by its frequency.
    population = []
    for bitstring, count in freq_dict.items():
        population.extend([bitstring] * count)
    
    total_count = len(population)
    if n_samples > total_count:
        raise ValueError("n_samples must be smaller than or equal to the total frequency count.")
    
    # Sample without replacement from the population.
    sampled = random.sample(population, n_samples)
    
    # Count the frequencies in the new sample.
    new_freq_dict = dict(Counter(sampled))
    return new_freq_dict

In [ ]:
if shot_wise:
    
    for exp in data:
        
        for provider, bs_ls in exp["metadata"]["backends"].items():
            __bs = []
            for bs in bs_ls:
                if bs not in __bs:
                    __bs.append(bs)
        
        b_len = len(__bs)
        d = {
            "circuit": exp["metadata"]["circuit"],
            "size": exp["metadata"]["circuit_size"],
            "seed": exp["metadata"]["seed"],
            "backends": "/".join(__bs),
            "backends_len": b_len,
        }
        
        block_shots = exp["metadata"]["shots"]["shots_block"]
        block_shots = int(block_shots)
        resampling_shots = block_shots // b_len
        resampling_shots = int(resampling_shots)
        rest = block_shots % b_len

        gt = normalise(exp["metadata"]["ground_truth"])
        
        last_cumulated_distribution = None
        
        _data = exp["individual"].copy()
        for i, s in enumerate(_data): 
            _d = d.copy()
            _d["it"] = i

            agg_dist = {}
            for provider, bs_ls in s.items():
                for b, r in bs_ls.items():
                    _dist = resample_freq_dict(r[0]['result'], resampling_shots)
                    for k, v in _dist.items():
                        agg_dist[k] = agg_dist.get(k, 0) + v
            if rest > 0:
                _dist = resample_freq_dict(r[0]['result'], rest)
                for k, v in _dist.items():
                    agg_dist[k] = agg_dist.get(k, 0) + v
            
            if last_cumulated_distribution is not None:
                for k in last_cumulated_distribution:
                    agg_dist[k] = agg_dist.get(k, 0) + last_cumulated_distribution[k]
                    
            _d["shots"] = sum(agg_dist.values())

            probs = normalise(agg_dist)
            
            _d["error"] = hellinger_distance(probs, gt)
            
            for d_name, d_fun in distances.items():
                _d[d_name] = d_fun(probs, gt)
            for m_name, m_fun in measures.items():
                _d[m_name] = m_fun(probs)
            
            if last_cumulated_distribution is None:
                for d_name in distances.keys():
                    _d["delta-"+d_name] = None
                for m_name in measures.keys():
                    _d["delta-"+m_name] = None
            else:
                last_probs = normalise(last_cumulated_distribution)
                
                for d_name, d_fun in distances.items():
                    _d["delta-"+d_name] = d_fun(last_probs, probs)
                for m_name, m_fun in measures.items():
                    _d["delta-"+m_name] = m_fun(last_probs) - m_fun(probs)
            
            ds.append(_d)
            
            last_cumulated_distribution = agg_dist.copy()

In [ ]:
df = pd.DataFrame(ds)
df

In [ ]:
# df["delta-hellinger-derivative"] = np.gradient(df["delta-hellinger"], df["it"])
# df["delta-stddev-derivative"] = np.gradient(df["delta-stddev"], df["it"])
# df

In [ ]:
def normalise_group(group):
    group = group.copy()
    group = (group - group.min()) / (group.max() - group.min())
    return group

def plot_together(grouped, metrics, normalised=False, logscale=False, legend=True):
# Loop over each metric to create one chart per metric
    for metric in metrics:
        plt.figure(figsize=(10, 6))
        
        # Loop over each group and plot the metric against shots
        for label, group in grouped:
            _group = group.copy()
            if normalised:
                _group[metric] = normalise_group(_group[metric])
            plt.plot(_group['shots'], _group[metric], label=label)
        
        plt.xlabel('Cumulative Shots')
        
        if metric == "error":
            plt.ylabel('TVD from Ground Truth')
        elif metric == "delta-TVD":
            plt.ylabel('Delta-TVD from Previous Iteration')
        else:
            plt.ylabel(metric)
        
        if logscale:
            plt.yscale('log')
        
        # plt.title(metric+(" (normalised)" if normalised else ""))
        if legend:
            plt.legend(title="Group", fontsize='small', loc='best')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        
def plot_together_metrics(grouped, metrics, normalised=False):
# Loop over each metric to create one chart per metric
    plt.figure(figsize=(10, 6))
    
    # Loop over each group and plot the metric against shots
    for label, group in grouped:
        _group = group.copy()
        for metric in metrics:
            if normalised:
                _group[metric] = normalise_group(_group[metric])
            plt.plot(_group['shots'], _group[metric], label=label)
    
    plt.xlabel('Shots')
    plt.ylabel('Metrics')
    plt.title(metrics+(" (normalised)" if normalised else ""))
    plt.legend(title="Group", fontsize='small', loc='best')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
        
def plot_separately(grouped, metrics, normalised=False):
# Loop over each metric to create one chart per metric
    for metric in metrics:
        # Loop over each group and plot the metric against shots
        for label, group in grouped:
            plt.figure(figsize=(10, 6))
            
            _group = group.copy()
            if normalised:
                _group[metric] = normalise_group(_group[metric])
            
            plt.plot(group['shots'], group[metric], label=label)
        
            plt.xlabel('Shots')
            plt.ylabel(metric)
            plt.title(metric+(" (normalised)" if normalised else ""))
            plt.legend(title="Group", fontsize='small', loc='best')
            plt.grid(True)
            plt.tight_layout()
            plt.show()
            
def plot_separately_metrics(grouped, metrics, normalised=False):
# Loop over each metric to create one chart per metric
    # Loop over each group and plot the metric against shots
    for label, group in grouped:
        plt.figure(figsize=(10, 6))
        
        _group = group.copy()
        
        for metric in metrics:
            if normalised:
                _group[metric] = normalise_group(_group[metric])
            
            plt.plot(group['shots'], group[metric], label=label)
    
        plt.xlabel('Shots')
        plt.ylabel('Metrics')
        plt.title(metrics+(" (normalised)" if normalised else ""))
        plt.legend(title="Group", fontsize='small', loc='best')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        
def plot_for_each_together(df, att, subgroups, metrics, normalised=False):
    
    atts = df[att].unique()
    for _att in atts:
        _df = df[df[att] == _att]
        grouped = _df.groupby(subgroups)
        
        # Loop over each metric to create one chart per metric
        for metric in metrics:
            plt.figure(figsize=(10, 6))
            
            # Loop over each group and plot the metric against shots
            for label, group in grouped:
                _group = group.copy()
                if normalised:
                    _group[metric] = normalise_group(_group[metric])
                plt.plot(_group['shots'], _group[metric], label=label)
            
            plt.xlabel('Shots')
            plt.ylabel(metric)
            plt.title(f"{att}: {_att} - {metric}"+(" (normalised)" if normalised else ""))
            plt.legend(title="Group", fontsize='small', loc='best')
            plt.grid(True)
            plt.tight_layout()
            plt.show()
        
def plot_for_each_separately(df, att, subgroups, metrics, normalised=False):
    atts = df[att].unique()
    for _att in atts:
        _df = df[df[att] == _att]
        grouped = _df.groupby(subgroups)
        
        # Loop over each group and plot the metric against shots
        for label, group in grouped:
            plt.figure(figsize=(10, 6))
            
            _group = group.copy()
            for metric in metrics:
                if normalised:
                    _group[metric] = normalise_group(_group[metric])
                
                plt.plot(group['shots'], group[metric], label=label)
            
            plt.xlabel('Shots')
            plt.ylabel(metric)
            plt.title(f"{att}: {_att} - {metric}"+(" (normalised)" if normalised else ""))
            plt.legend(title="Group", fontsize='small', loc='best')
            plt.grid(True)
            plt.tight_layout()
            plt.show()

In [ ]:
_df = df.copy()

In [ ]:
separate = False

In [ ]:
metrics = ["error"] + ["delta-"+m for m in measures.keys()] + ["delta-"+d for d in distances.keys()]#  + list(measures.keys()) + list(distances.keys())

In [ ]:
legend = False

plot_together(df.groupby(['backends', 'circuit', 'size', 'seed']), metrics, logscale=False, legend=legend)
plot_together(df.groupby(['backends', 'circuit', 'size', 'seed']), metrics, normalised=True, logscale=False, legend=legend)

In [ ]:
# other_metrics = ["hellinger", "stddev", "gt-stddev"]
# plot_together(df.groupby(['backends', 'circuit', 'size']), other_metrics)
# plot_together(df.groupby(['backends', 'circuit', 'size']), other_metrics, normalised=True)
# plot_together_metrics(df.groupby(['backends', 'circuit', 'size']), other_metrics)
# plot_together_metrics(df.groupby(['backends', 'circuit', 'size']), other_metrics, normalised=True)


In [ ]:
plot_for_each_together(df, 'backends', ['circuit', 'size', 'seed'], ["error"])
plot_for_each_together(df, 'backends', ['circuit', 'size', 'seed'], ["error"], normalised=True)

In [ ]:
#convert circuit to string
df["_circuit"] = df["circuit"].astype(str) + " " + df["size"].astype(str)

plot_for_each_together(df, "_circuit", ["backends", "seed"], metrics)
plot_for_each_together(df, "_circuit", ["backends", "seed"], metrics, normalised=True)

In [ ]:
plot_for_each_together(df, "size", ["backends", "circuit", "seed"], metrics)
plot_for_each_together(df, "size", ["backends", "circuit", "seed"], metrics, normalised=True)

In [ ]:
plot_together(df.groupby(['backends', 'circuit', 'size', 'seed']), ["stddev", "hellinger"])
plot_together(df.groupby(['backends', 'circuit', 'size', 'seed']), ["stddev", "hellinger"], normalised=True)